In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_spectrum_freq_S5_20261005_170000.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_spectrum_freq_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
all_runs = pd.read_csv(RESULT_CSV).sort_values("f_vco").reset_index(drop=True)
for col in ("time_jitter_common_ps", "time_jitter_trace_common_ps", "carrier_dbm"):  # runs before these columns
    if col not in all_runs:
        all_runs[col] = np.nan
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["f_vco", "kvco_target", "kind"])
offsets = [c[len("L_vco_"):] for c in all_runs.columns if c.startswith("L_vco_")]

runs = all_runs[all_runs["locked"].fillna(0) == 1].reset_index(drop=True)
skipped = all_runs[all_runs["locked"].fillna(0) != 1]
print(f"{RESULT_CSV.name}: {len(all_runs)} frequencies, {len(runs)} locked "
      f"({(runs['status'] == 'ok').sum()} measured ok), requested |Kvco| {all_runs['kvco_request'].iloc[0]}")
if len(skipped):
    print("not locked: " + ", ".join(f"{f / 1e6:g} MHz" for f in skipped["f_vco"]))

pd.DataFrame({
    "f_vco [MHz]": all_runs["f_vco"] / 1e6,
    "f_pad [MHz]": all_runs["f_out_meas"] / 1e6,  # analyzer centre frequency = carrier on the pad
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                if not np.isnan(r.vco_tune_coarse) else "-" for r in all_runs.itertuples()],
    "|Kvco| [MHz/V]": all_runs["kvco_pred"] / 1e6,
    "possible [MHz/V]": [f"{a / 1e6:.0f}-{b / 1e6:.0f}" for a, b in zip(all_runs["kvco_min"], all_runs["kvco_max"])],
    "Vctrl pred [V]": all_runs["vctrl_pred"],
    "f_ref [MHz]": all_runs["f_ref"] / 1e6,
    "N": all_runs["total_div"],
    "locked": all_runs["locked"],
    "carrier [dBm]": all_runs["carrier_dbm"],
    "P [mW]": all_runs["pll_p"] * 1e3,
    "jitter markers [ps]": all_runs["time_jitter_ps"],
    "jitter curve [ps]": all_runs["time_jitter_trace_ps"],
    "jitter noise [ps]": all_runs["time_jitter_noise_ps"],
    "jitter spurs [ps]": all_runs["time_jitter_spur_ps"],
    "status": all_runs["status"],
}).round(3)

pll_spectrum_freq_S5_20261007_114859.csv: 8 frequencies, 7 locked (7 measured ok), requested |Kvco| 2000000000.0
not locked: 300 MHz


,f_vco [MHz],f_pad [MHz],setting,|Kvco| [MHz/V],possible [MHz/V],Vctrl pred [V],f_ref [MHz],N,locked,carrier [dBm],P [mW],jitter markers [ps],jitter curve [ps],jitter noise [ps],jitter spurs [ps],status
0,300.0,NaN,c7 min 14 max 15,2000.235,317-7689,0.565,2.344,4,0,NaN,0.278,NaN,NaN,NaN,NaN,not_locked
1,500.0,125.00,c13 min 11 max 8,2005.482,415-12123,0.639,3.906,4,1,-2.919,0.364,1442.832,1386.172,1306.145,464.174,ok
2,750.0,93.75,c10 min 9 max 2,2004.078,332-17527,0.622,5.859,8,1,-0.266,0.432,794.896,841.545,807.739,236.126,ok
3,1000.0,125.00,c6 min 8 max 10,1995.819,62-21142,0.613,7.812,8,1,-1.516,0.460,1181.863,959.810,861.117,423.925,ok
4,1500.0,93.75,c0 min 11 max 15,1938.995,1849-27460,0.617,11.719,16,1,-1.885,0.471,1391.385,1545.280,1474.487,462.362,ok
5,2000.0,125.00,c0 min 9 max 0,1938.276,1179-28234,0.674,15.625,16,1,-3.807,0.544,1238.118,1713.065,1644.611,479.424,ok
6,3000.0,93.75,c0 min 8 max 12,15020.646,15021-33302,0.559,23.438,32,1,0.685,0.701,66.556,220.805,217.270,39.353,ok
7,4000.0,125.00,c0 min 8 max 0,28039.258,28039-52997,0.553,31.250,32,1,0.433,0.880,30.172,69.438,63.663,27.725,ok


In [ ]:
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

INK, GRID = "#52514e", "#e4e3df"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
FREQ_AXIS = dict(type="log", tickvals=[100, 200, 500, 1e3, 2e3, 5e3, 1e4],
                 ticktext=["100 MHz", "200 MHz", "500 MHz", "1 GHz", "2 GHz", "5 GHz", "10 GHz"],
                 minor=dict(showgrid=True, gridcolor="#f1f0ed"))
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 300  # points per plotted phase-noise curve (spread over F_start-F_stop); None = every measured point

# One colour per frequency, in sweep order (not cycled: more than 8 frequencies -> split the run).
COLORS = {f: PALETTE[i] for i, f in enumerate(runs["f_vco"])}
NAMES = {r.f_vco: f"{r.f_vco / 1e6:g} MHz (ref {r.f_ref / 1e6:.4g} MHz): |Kvco| {r.kvco_pred / 1e6:.0f} MHz/V, c{r.vco_tune_coarse:.0f} "
                  f"min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}, carrier {r.carrier_dbm:.1f} dBm"
         for r in runs.itertuples()}
ok = runs[runs["status"] == "ok"]
HOVER_DATA = np.c_[ok["vco_tune_coarse"], ok["vco_current_min"], ok["vco_current_max"], ok["kvco_pred"] / 1e6,
                   ok["carrier_dbm"], ok["f_ref"] / 1e6]
HOVER_SETTING = ("ref %{customdata[5]:.4g} MHz, c%{customdata[0]:.0f} min %{customdata[1]:.0f} max %{customdata[2]:.0f}, "
                 "|Kvco| %{customdata[3]:.0f} MHz/V, carrier %{customdata[4]:.1f} dBm")


def trace_of(f_vco, kind):
    return traces[(traces["f_vco"] == f_vco) & (traces["kind"] == kind)]


def offset_axis(f_start, f_stop, f_axis):
    """x-axis settings for an offset plot shown from f_start to f_stop [Hz]; f_axis "log" or "lin"."""
    if f_axis == "log":
        return dict(OFFSET_AXIS, range=[np.log10(f_start), np.log10(f_stop)])
    return dict(type="linear", range=[f_start, f_stop], ticksuffix="Hz", exponentformat="SI")


def in_range(df, f_start, f_stop):
    """Rows of df with the offset (column freq) between f_start and f_stop, so the y axis fits the shown band."""
    return df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)]


def binned(df, f_start, f_stop, f_axis):
    """The curve in df (columns freq, level_dbm) between f_start and f_stop, averaged into N_BINS bins spaced for
    f_axis, in linear power (pll_util.bin_phase_noise): display only, the jitter uses the measured points."""
    df = in_range(df, f_start, f_stop).sort_values("freq")
    if N_BINS is None or len(df) < 2:
        return df
    if f_axis == "log":
        edges = np.geomspace(f_start, f_stop, N_BINS + 1)
    else:
        edges = np.linspace(f_start, f_stop, N_BINS + 1)
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], edges)
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [ ]:
# |Kvco| reached per frequency (dots: locked filled, not locked open) against the requested one (dashed), with the
# range the lookup table offers at that frequency (bar).
fig = go.Figure()
fig.add_bar(x=all_runs["f_vco"] / 1e6, y=(all_runs["kvco_max"] - all_runs["kvco_min"]) / 1e6,
            base=all_runs["kvco_min"] / 1e6, width=all_runs["f_vco"] / 1e6 * 0.12, marker_color="#d6e4f5",
            name="possible in the table", customdata=np.c_[all_runs["kvco_min"] / 1e6, all_runs["kvco_max"] / 1e6],
            hovertemplate="%{x:g} MHz: %{customdata[0]:.0f}-%{customdata[1]:.0f} MHz/V possible<extra></extra>")
for locked, symbol, name in ((1, "circle", "chosen setting (locked)"), (0, "circle-open", "chosen setting (not locked)")):
    d = all_runs[all_runs["locked"].fillna(0) == locked]
    fig.add_scatter(x=d["f_vco"] / 1e6, y=d["kvco_pred"] / 1e6, mode="markers", name=name,
                    marker=dict(symbol=symbol, size=11, color=PALETTE[0], line=dict(width=2)),
                    hovertemplate="%{x:g} MHz: |Kvco| %{y:.0f} MHz/V<extra></extra>")
k_req = float(all_runs["kvco_request"].iloc[0])
fig.add_hline(y=k_req / 1e6, line=dict(color=INK, width=1, dash="dash"),
              annotation_text=f"requested {k_req / 1e6:.0f} MHz/V", annotation_position="top left")
fig.update_layout(title="|Kvco| reached per frequency", template="plotly_white", height=480, barmode="overlay",
                  legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="f_vco", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="|Kvco| [MHz/V]", type="log", gridcolor=GRID)
save(fig, "kvco_vs_f")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_freq_S5_20261007_114859_kvco_vs_f.html and .png


In [ ]:
F_start, F_stop = 1e3, 100e6  # offset range shown [Hz]
f_axis = "log"  # "log" or "lin"

# Phase noise referred to the VCO per frequency: full curve (line, binned to N_BINS points: linear power average,
# display only), detected spurs (x at their measured peak, hover: power on the pad).
fig = go.Figure()
for r in ok.itertuples():
    n_db = 20 * np.log10(r.total_div)
    lt_raw = in_range(trace_of(r.f_vco, "L_trace"), F_start, F_stop)  # spur peaks: from the measured points
    lt = binned(trace_of(r.f_vco, "L_trace"), F_start, F_stop, f_axis)
    sp = in_range(trace_of(r.f_vco, "spur"), F_start, F_stop)
    if lt.empty:
        continue
    fig.add_scatter(x=lt["freq"], y=lt["level_dbm"] + n_db, mode="lines", line=dict(color=COLORS[r.f_vco], width=2),
                    name=NAMES[r.f_vco], legendgroup=NAMES[r.f_vco],
                    hovertemplate="%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    if not sp.empty:
        fig.add_scatter(x=sp["freq"], y=np.interp(sp["freq"], lt_raw["freq"], lt_raw["level_dbm"]) + n_db,
                        mode="markers", marker=dict(symbol="x", size=10, color=COLORS[r.f_vco], line=dict(width=1)),
                        showlegend=False, legendgroup=NAMES[r.f_vco],
                        customdata=np.c_[sp["level_dbm"], sp["freq"] / r.f_ref],
                        hovertemplate=("spur at %{x:.4s}Hz = %{customdata[1]:.2f} x ref: %{customdata[0]:.1f} dBc on "
                                       "the pad<extra></extra>"))
fig.update_layout(title="Phase noise referred to the VCO (pad + 20 log10 N) per frequency (x: spurs, on the pad)", template="plotly_white",
                  height=580, legend=dict(orientation="h", y=-0.2), hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **offset_axis(F_start, F_stop, f_axis))
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

In [ ]:
# RMS jitter vs f_vco: full curve (total, noise only, spurs) and the markers.
fig = go.Figure()
for col, name, color, dash in (("time_jitter_trace_ps", "full curve, total", PALETTE[0], "solid"),
                               ("time_jitter_noise_ps", "full curve, noise only", PALETTE[2], "solid"),
                               ("time_jitter_spur_ps", "full curve, spurs", PALETTE[3], "dot"),
                               ("time_jitter_ps", "markers", PALETTE[1], "dash"),
                               ("time_jitter_trace_common_ps", "full curve, common band", PALETTE[4], "dashdot")):
    fig.add_scatter(x=ok["f_vco"] / 1e6, y=ok[col], mode="lines+markers", name=name,
                    line=dict(color=color, width=2, dash=dash), marker=dict(size=9), customdata=HOVER_DATA,
                    hovertemplate=f"{name}: %{{y:.1f}} ps at %{{x:g}} MHz<br>{HOVER_SETTING}<extra></extra>")
fig.update_layout(title=f"RMS time jitter vs f_vco (same on the pad and at the VCO), integrated {float(offsets[0]):g} Hz - {float(offsets[-1]) / 1e6:g} MHz",
                  template="plotly_white", height=480, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="f_vco", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="RMS time jitter [ps]", type="log", gridcolor=GRID)
save(fig, "jitter_vs_f")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_freq_S5_20261007_114859_jitter_vs_f.html and .png


In [ ]:
# PLL supply power and carrier level on the pad vs f_vco.
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["PLL supply power", "Carrier level on the pad (= PLL output)"])
fig.add_scatter(x=ok["f_vco"] / 1e6, y=ok["pll_p"] * 1e3, mode="lines+markers", showlegend=False, row=1, col=1,
                line=dict(color=PALETTE[0], width=2), marker=dict(size=9), customdata=HOVER_DATA,
                hovertemplate=f"%{{y:.3f}} mW at %{{x:g}} MHz<br>{HOVER_SETTING}<extra></extra>")
fig.add_scatter(x=ok["f_vco"] / 1e6, y=ok["carrier_dbm"], mode="lines+markers", showlegend=False, row=2, col=1,
                line=dict(color=PALETTE[1], width=2), marker=dict(size=9), customdata=np.c_[HOVER_DATA, ok["total_div"]],
                hovertemplate=f"%{{y:.1f}} dBm at %{{x:g}} MHz (/%{{customdata[6]:.0f}})<br>{HOVER_SETTING}<extra></extra>")
fig.update_layout(template="plotly_white", height=600)
fig.update_xaxes(gridcolor=GRID, **FREQ_AXIS)
fig.update_xaxes(title="f_vco", row=2)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(title="power [mW]", rangemode="tozero", row=1)
fig.update_yaxes(title="carrier [dBm]", row=2)
save(fig, "power_carrier_vs_f")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_freq_S5_20261007_114859_power_carrier_vs_f.html and .png
